# Pytorch CNN Example

From [YouTube - Pytorch CNN Example](https://www.youtube.com/watch?v=wnK3uWv_WkU&list=PLhhyoLH6IjfxeoooqP9rhU3HJIAVAJ3Vz&index=4)

### CNN formula for determining the number of output channels

$$n_{out} = \left[\frac{n_{in} + 2p - k}{s}\right]+1$$

* $n_{in}$: Number of input features
* $n_{out}$: Number of output features
* $k$: Convolution kernel size
* $p$: Convolution padding size
* $s$: Convolution stride size

In [1]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms

In [ ]:

# Simple CNN
class CNN(nn.Module):
    def __init__(self, in_channels = 1, num_classes = 10):
        super(CNN, self).__init__()
        # Same convolution (same dimensions as input)
        # 28 in features -> 28 out features
        self.conv1 = nn.Conv2d(
            in_channels=in_channels, 
            out_channels=8, 
            kernel_size=(3,3), 
            stride=(1,1), 
            padding=(1,1)
            ) 
        
        # Halve the dimension size by mean pooling
        self.pool = nn.MaxPool2d(
            kernel_size=(2,2), 
            stride=(2,2)
            )
        
        # 8 features in, 16 features out
        self.conv2 = nn.Conv2d(
            in_channels=8, 
            out_channels=16, 
            kernel_size=(3,3), 
            stride=(1,1), 
            padding=(1,1)
            ) 
        
        self.fc1 = nn.Linear(16*7*7, num_classes)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))       # (B, 1, 28, 28) -> (B, 8, 28, 28)
        x = self.pool(x)                # (B, 8, 28, 28) -> (B, 8, 14, 14)
        x = F.relu(self.conv2(x))       # (B, 8, 14, 14) -> (B, 16, 14, 14)
        x = self.pool(x)                # (B, 16, 14, 14) -> (B, 16, 7, 7)
        x = x.reshape(x.shape[0], -1)   # (B, 16, 7, 7) -> (B, 784)
        x = self.fc1(x)                 # (B, 784) -> (B, num_classes)
        
        return x

model = CNN()
x = torch.randn(64, 1, 28, 28)
print(x.shape)
print(model(x).shape)

torch.Size([64, 1, 28, 28])
torch.Size([64, 10])


In [18]:
model = CNN()
x = torch.randn(64, 1, 28, 28)
print(x.shape)
x = F.relu(CNN().conv1(x))
print(x.shape)
x = CNN().pool(x)
print(x.shape)
x = F.relu(CNN().conv2(x))
x = CNN().pool(x)
print(x.shape)
x = x.reshape(x.shape[0], -1)
print(x.shape)
x = CNN().fc1(x)
print(x.shape)


torch.Size([64, 1, 28, 28])
torch.Size([64, 8, 28, 28])
torch.Size([64, 8, 14, 14])
torch.Size([64, 16, 7, 7])
torch.Size([64, 784])
torch.Size([64, 10])


In [8]:
model

CNN(
  (conv1): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=784, out_features=10, bias=True)
)

In [7]:
# Set Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hyperparameters
in_channel = 1
num_classes = 10
learning_rate = 0.001
batch_size = 64
num_epochs = 5

# Load Data
train_dataset = datasets.MNIST(
    root='dataset/',                    # dataset for saving the data
    train=True,                         # Use the training set?
    transform=transforms.ToTensor(),    # Transform the data to tensors
    download=True                       # download if not present in the `dataset/` directory
    )

test_dataset = datasets.MNIST(
    root='dataset/',                    # dataset for saving the data
    train=False,                         # Use the training set?
    transform=transforms.ToTensor(),    # Transform the data to tensors
    download=True                       # download if not present in the `dataset/` directory
    )


train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

# Initialize Network
model = CNN().to(device)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Check accuracy on training and test
def check_accuracy(loader, model, epoch):
    if loader.dataset.train:
        print("Testing accuracy on training data")
    
    num_correct = 0
    num_samples = 0
    model.eval()
    
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)
            
            scores = model(x)
            
            # 64x10
            _, predictions = scores.max(1) # index of max value
            num_correct += (predictions == y).sum()   # Number of correct predictions
            num_samples += predictions.size(0)        # Number of samples
            
        print(f"Epoch {epoch+1}: {num_correct:,} / {num_samples:,} | Accuracy {float(num_correct)/float(num_samples)*100:.2f}")
            
            

# Train Network
for epoch in range(num_epochs):
    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device=device)
        targets = targets.to(device=device)
                
        # Forward
        scores = model(data)
        loss = criterion(scores, targets)
        
        # Backward
        optimizer.zero_grad() # Set all the gradients to 0 for each batch
        loss.backward()
        
        # Gradient descent or adam step
        optimizer.step()
         
    check_accuracy(test_loader, model, epoch)


Epoch 1: 9,700 / 10,000 | Accuracy 97.00
Epoch 2: 9,763 / 10,000 | Accuracy 97.63
Epoch 3: 9,774 / 10,000 | Accuracy 97.74
Epoch 4: 9,823 / 10,000 | Accuracy 98.23
Epoch 5: 9,826 / 10,000 | Accuracy 98.26
